# IAN 601 — Module 13: Statistics & Visualization

Statistics and visualization are two complementary ways to understand data.

- **Statistics** summarize data with numbers.
- **Visualizations** help us see shape, spread, differences, trends, and relationships.

A good analysis usually uses both.

This notebook is designed as a complete, self-contained learning resource.

## Learning goals

By the end of this notebook, you should be able to:

1. describe numeric data with count, mean, median, standard deviation, quartiles, minimum, and maximum;
2. explain the difference between mean and median;
3. measure spread with range, standard deviation, and IQR;
4. use histograms and boxplots to study distributions;
5. compare groups with summary statistics and grouped plots;
6. use bar charts for categorical comparisons;
7. use scatter plots to study relationships between numeric variables;
8. calculate and interpret correlation carefully;
9. explain the basic idea of a confidence interval;
10. summarize time-based data with a line chart;
11. smooth a noisy time series with a rolling average;
12. combine numerical and visual evidence into a short analytical interpretation.

## How to use this notebook

Run the cells from top to bottom the first time.

For each section:

1. read the explanation before running the code;
2. predict what you expect to see;
3. run the code;
4. inspect the result;
5. change one option or value;
6. run it again and explain what changed.

The dataset loads directly from GitHub, so it works in Google Colab without uploading a file manually.

### Expandable syntax guides

Each major analytical section includes a collapsed **Syntax & parameters** box. Open it for function signatures, chart options, statistical details, related methods, and interpretation cautions.

## 1. Import the libraries

We will use:

- **pandas** for tables and summaries,
- **NumPy** for numerical calculations,
- **Matplotlib** for charts.

<details>
<summary><strong>Syntax & parameters — common statistics libraries</strong></summary>

<pre><code class="language-python">import pandas as pd
import numpy as np
import matplotlib.pyplot as plt</code></pre>

<ul>
<li><code>pandas</code> — tabular data and group summaries.</li>
<li><code>numpy</code> — numerical operations.</li>
<li><code>matplotlib.pyplot</code> — plotting interface.</li>
</ul>

<p>Other libraries you may encounter later include <code>scipy.stats</code> for statistical distributions/tests and <code>statsmodels</code> for statistical models.</p>
</details>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2. Load the dataset

The dataset contains 400 order records.

Each row represents one order.

Important columns:

- `order_id` — order identifier
- `order_date` — date of the order
- `region` — geographic region
- `channel` — sales channel
- `amount` — order amount
- `satisfaction` — satisfaction score from 1 to 5

<details>
<summary><strong>Syntax & parameters — read_csv() with dates</strong></summary>

<pre><code class="language-python">pd.read_csv(
    "file.csv",
    parse_dates=["order_date"]
)</code></pre>

<p><code>parse_dates</code> tells Pandas to convert selected columns to datetime while loading.</p>

<p>Useful alternatives:</p>
<pre><code class="language-python">df["order_date"] = pd.to_datetime(
    df["order_date"],
    errors="coerce"
)</code></pre>

<p>Other useful <code>read_csv()</code> parameters include <code>usecols</code>, <code>dtype</code>, <code>na_values</code>, and <code>nrows</code>.</p>
</details>

In [ ]:
data_url = "https://raw.githubusercontent.com/ali2449/IAN-601-Introduction-to-Data-Analytics-Methods-and-Approaches/main/IAN601_Module13_Statistics_Visualization_400.csv"

orders = pd.read_csv(
    data_url,
    parse_dates=["order_date"]
)

orders.head()

## 3. Inspect before analyzing

Before calculating statistics, inspect the structure of the table.

Useful questions include:

- How many rows and columns are there?
- Which variables are numeric?
- Which variables are categorical?
- Are dates stored as dates?
- Are values missing?

<details>
<summary><strong>Syntax & parameters — inspection before statistics</strong></summary>

<pre><code class="language-python">df.shape
df.columns
df.dtypes
df.info()
df.head()
df.sample(5)
df.isna().sum()</code></pre>

<p>Before calculating statistics, confirm that numeric variables are numeric, dates are datetime values, and categorical labels are consistent.</p>

<p>Use <code>select_dtypes(include="number")</code> to select numeric columns and <code>select_dtypes(include=["object", "string", "category"])</code> for many categorical columns.</p>
</details>

In [ ]:
print("Shape:", orders.shape)
print()
print("Columns:")
print(orders.columns.tolist())
print()
print("Data types:")
print(orders.dtypes)

In [ ]:
print("Missing values:")
print(orders.isna().sum())

### Variable types matter

Different variables support different kinds of analysis.

| Variable | Type | Useful analysis |
|---|---|---|
| `region` | categorical | counts, group comparisons, bar charts |
| `channel` | categorical | counts, proportions, bar charts |
| `amount` | numeric | mean, median, histogram, boxplot |
| `satisfaction` | ordinal score | counts, averages, scatter plot, correlation with caution |
| `order_date` | date/time | trends over time |

Choosing an appropriate statistic or chart begins with understanding the variable type.

## 4. Descriptive statistics

Descriptive statistics summarize what is present in the dataset.

For a numeric variable, common summaries include:

- **count** — number of non-missing observations;
- **mean** — arithmetic average;
- **standard deviation** — typical spread around the mean;
- **minimum** — smallest value;
- **25th percentile** — value below which about 25% of observations fall;
- **median** — middle value;
- **75th percentile** — value below which about 75% of observations fall;
- **maximum** — largest value.

<details>
<summary><strong>Syntax & parameters — describe() and summary statistics</strong></summary>

<pre><code class="language-python">s.describe()
df.describe()
df.describe(include="all")</code></pre>

<p><strong>Individual statistics</strong></p>
<pre><code class="language-python">s.count()
s.mean()
s.median()
s.std()
s.var()
s.min()
s.max()
s.quantile([0.25, 0.5, 0.75])</code></pre>

<p>Pandas <code>std()</code> uses sample standard deviation by default with <code>ddof=1</code>.</p>
</details>

In [ ]:
orders["amount"].describe()

In [ ]:
print("Mean:", round(orders["amount"].mean(), 2))
print("Median:", round(orders["amount"].median(), 2))
print("Standard deviation:", round(orders["amount"].std(), 2))
print("Minimum:", round(orders["amount"].min(), 2))
print("Maximum:", round(orders["amount"].max(), 2))

## 5. Mean versus median

The **mean** is the arithmetic average.

The **median** is the middle observation after sorting the data.

The mean uses every value and is therefore more sensitive to unusually large or small observations.

The median is more resistant to extreme values.

<details>
<summary><strong>Syntax & concepts — measures of center</strong></summary>

<pre><code class="language-python">s.mean()
s.median()
s.mode()</code></pre>

<ul>
<li><strong>Mean</strong> — arithmetic average; sensitive to extreme values.</li>
<li><strong>Median</strong> — middle ordered value; resistant to extreme values.</li>
<li><strong>Mode</strong> — most frequent value; can have more than one result.</li>
</ul>

<p>For skewed variables such as income or transaction size, reporting both mean and median is often more informative than reporting only one.</p>
</details>

In [ ]:
mean_amount = orders["amount"].mean()
median_amount = orders["amount"].median()

print("Mean amount:", round(mean_amount, 2))
print("Median amount:", round(median_amount, 2))
print("Difference:", round(mean_amount - median_amount, 2))

### Why compare them?

If the mean and median are close, the distribution may be fairly balanced.

If they are noticeably different, the data may be skewed or influenced by unusual observations.

This is only a clue. A visualization helps us understand the shape more directly.

### Small example: effect of an extreme value

The following example shows why the mean and median react differently.

In [ ]:
example = pd.Series([50, 55, 60, 65, 70])

print("Original mean:", example.mean())
print("Original median:", example.median())

example_with_extreme = pd.Series([50, 55, 60, 65, 300])

print()
print("After adding an extreme value:")
print("Mean:", example_with_extreme.mean())
print("Median:", example_with_extreme.median())

The extreme value changes the mean much more than the median.

That is why analysts often report both.

## 6. Measuring spread

Center is only part of the story.

Two datasets can have the same mean but very different variability.

Useful measures of spread include:

- **range** — maximum minus minimum;
- **standard deviation** — typical distance from the mean;
- **interquartile range, or IQR** — width of the middle 50% of the data.

<details>
<summary><strong>Syntax & concepts — range, variance, standard deviation, and IQR</strong></summary>

<pre><code class="language-python">data_range = s.max() - s.min()
variance = s.var()
std = s.std()

q1 = s.quantile(0.25)
q3 = s.quantile(0.75)
iqr = q3 - q1</code></pre>

<p><strong>Interpretation</strong></p>
<ul>
<li>Range uses only the two extremes.</li>
<li>Variance is in squared units.</li>
<li>Standard deviation is in the original units.</li>
<li>IQR measures the width of the middle 50% and is resistant to extremes.</li>
</ul>
</details>

In [ ]:
amount = orders["amount"]

data_range = amount.max() - amount.min()
std_amount = amount.std()

q1 = amount.quantile(0.25)
q3 = amount.quantile(0.75)
iqr = q3 - q1

print("Range:", round(data_range, 2))
print("Standard deviation:", round(std_amount, 2))
print("Q1:", round(q1, 2))
print("Q3:", round(q3, 2))
print("IQR:", round(iqr, 2))

## 7. Histogram: understand a distribution

A histogram groups numeric values into intervals called **bins**.

A histogram helps answer questions such as:

- Where are most observations concentrated?
- Is the distribution approximately symmetric?
- Is one side longer than the other?
- Are there multiple clusters?
- Are there gaps?
- Are there unusual observations?

<details>
<summary><strong>Syntax & parameters — plt.hist()</strong></summary>

<pre><code class="language-python">plt.hist(
    x,
    bins=20,
    range=None,
    density=False,
    edgecolor="black"
)</code></pre>

<ul>
<li><code>x</code> — numeric values.</li>
<li><code>bins</code> — number of bins or explicit bin boundaries.</li>
<li><code>range</code> — optional minimum/maximum shown.</li>
<li><code>density=True</code> — scale to density rather than counts.</li>
<li><code>alpha</code> — transparency.</li>
</ul>

<p>Different bin choices can change the appearance of a histogram. Use enough bins to reveal shape without making the chart unnecessarily noisy.</p>
</details>

In [ ]:
plt.figure(figsize=(8, 4))

plt.hist(
    orders["amount"],
    bins=20,
    edgecolor="black"
)

plt.xlabel("Order amount")
plt.ylabel("Frequency")
plt.title("Distribution of Order Amounts")
plt.tight_layout()
plt.show()

### How to read the histogram

Do not focus only on one bar.

Look at the overall shape:

1. Where is the center?
2. How wide is the distribution?
3. Where are most values concentrated?
4. Does one side extend farther?
5. Are there isolated areas?

The histogram adds information that a mean alone cannot provide.

## 8. Boxplot: summarize center and spread

A boxplot is a compact summary of a distribution.

Important parts include:

- the line inside the box — **median**;
- bottom of the box — **Q1**;
- top of the box — **Q3**;
- height of the box — **IQR**;
- whiskers — typical range outside the box;
- isolated points — possible unusual observations.

<details>
<summary><strong>Syntax & parameters — plt.boxplot()</strong></summary>

<pre><code class="language-python">plt.boxplot(
    x,
    vert=False,
    showmeans=False,
    showfliers=True
)</code></pre>

<ul>
<li><code>vert</code> — vertical or horizontal orientation.</li>
<li><code>showmeans</code> — display the mean marker.</li>
<li><code>showfliers</code> — display points beyond the whiskers.</li>
<li><code>whis</code> — whisker rule; default is commonly 1.5 × IQR.</li>
<li><code>labels</code> / <code>tick_labels</code> — group labels depending on Matplotlib version.</li>
</ul>

<p>A boxplot flags possible outliers; it does not determine whether they should be removed.</p>
</details>

In [ ]:
plt.figure(figsize=(8, 3.5))

plt.boxplot(
    orders["amount"],
    vert=False
)

plt.xlabel("Order amount")
plt.title("Boxplot of Order Amounts")
plt.tight_layout()
plt.show()

### Boxplots and outliers

A point beyond the whiskers is not automatically an error.

It could be:

- a data-entry mistake,
- a legitimate large transaction,
- a special case,
- evidence of a subgroup.

Outliers should be investigated before they are removed.

## 9. Compare distributions across regions

Sometimes we want to compare the same numeric variable across several groups.

A grouped boxplot makes it easier to compare:

- median,
- spread,
- possible unusual observations,
- differences between groups.

<details>
<summary><strong>Syntax & parameters — grouped boxplots</strong></summary>

<pre><code class="language-python">groups = [
    df.loc[df["region"] == region, "amount"]
    for region in region_order
]

plt.boxplot(
    groups,
    labels=region_order
)</code></pre>

<p>Pandas also offers:</p>
<pre><code class="language-python">df.boxplot(
    column="amount",
    by="region"
)</code></pre>

<p>Use a consistent group order when comparing charts so readers do not have to mentally reorder categories.</p>
</details>

In [ ]:
region_order = ["North", "South", "East", "West"]

region_amounts = [
    orders.loc[
        orders["region"] == region,
        "amount"
    ]
    for region in region_order
]

plt.figure(figsize=(8, 4))

plt.boxplot(
    region_amounts,
    labels=region_order
)

plt.xlabel("Region")
plt.ylabel("Order amount")
plt.title("Order Amount by Region")
plt.tight_layout()
plt.show()

### What should you look for?

Compare:

- the position of the median line;
- the size of each box;
- the whiskers;
- possible unusual points.

A visual comparison helps us decide which numerical summaries to calculate next.

## 10. Compare groups with summary statistics

Now calculate the statistics behind the grouped visualization.

<details>
<summary><strong>Syntax & parameters — grouped statistical summaries</strong></summary>

<pre><code class="language-python">df.groupby("region")["amount"].agg(
    ["count", "mean", "median", "std"]
)</code></pre>

<p><strong>Named version</strong></p>
<pre><code class="language-python">df.groupby("region", as_index=False).agg(
    n=("amount", "count"),
    mean_amount=("amount", "mean"),
    median_amount=("amount", "median"),
    std_amount=("amount", "std")
)</code></pre>

<p>Always consider sample size beside mean and variability. A group with very few observations may have an unstable average.</p>
</details>

In [ ]:
region_summary = (
    orders
    .groupby("region")["amount"]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)

region_summary

### Interpreting the table

When comparing groups, avoid looking only at the mean.

Consider:

- **count** — how many observations support the estimate;
- **mean** — average value;
- **median** — middle value;
- **standard deviation** — variability within the group.

A group can have a higher mean and also greater variability.

## 11. Bar charts for categorical comparisons

Bar charts are useful for comparing categories.

First calculate a summary value for each category. Then plot those category summaries.

Here we compare mean order amount by region.

<details>
<summary><strong>Syntax & parameters — bar charts</strong></summary>

<pre><code class="language-python">summary.plot(
    kind="bar",
    figsize=(8, 4)
)</code></pre>

<p>Or with Matplotlib:</p>
<pre><code class="language-python">plt.bar(categories, values)</code></pre>

<p>Useful options include <code>width</code>, <code>alpha</code>, and horizontal bars with <code>plt.barh()</code>.</p>

<p>Bar charts should usually start from zero when bar length represents magnitude, because truncating the axis can exaggerate differences.</p>
</details>

In [ ]:
regional_mean = (
    orders
    .groupby("region")["amount"]
    .mean()
    .sort_values()
)

regional_mean

In [ ]:
plt.figure(figsize=(8, 4))

regional_mean.plot(kind="bar")

plt.xlabel("Region")
plt.ylabel("Mean order amount")
plt.title("Average Order Amount by Region")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Bar chart versus histogram

These charts answer different questions.

- **Histogram:** What does the distribution of one numeric variable look like?
- **Bar chart:** How does a summary value compare across categories?

Choosing the wrong chart can make a correct analysis harder to understand.

## 12. Count categorical values

Before comparing categories, check how many observations belong to each one.

<details>
<summary><strong>Syntax & parameters — value_counts() and crosstab()</strong></summary>

<pre><code class="language-python">s.value_counts(
    normalize=False,
    dropna=True,
    sort=True
)</code></pre>

<p>Set <code>normalize=True</code> for proportions and <code>dropna=False</code> to include missing values.</p>

<p><strong>Cross-tabulation</strong></p>
<pre><code class="language-python">pd.crosstab(
    df["region"],
    df["channel"],
    normalize=False,
    margins=True
)</code></pre>

<p>Use <code>normalize="index"</code> or <code>normalize="columns"</code> to compare proportions within rows or columns.</p>
</details>

In [ ]:
print("Orders by region:")
print(orders["region"].value_counts())

print()
print("Orders by channel:")
print(orders["channel"].value_counts())

If one category contains far fewer observations than the others, comparisons may be less stable and should be interpreted carefully.

## 13. Scatter plots for numeric relationships

A scatter plot examines the relationship between two numeric variables.

Each point represents one observation.

Here:

- x-axis = order amount
- y-axis = satisfaction score

<details>
<summary><strong>Syntax & parameters — plt.scatter()</strong></summary>

<pre><code class="language-python">plt.scatter(
    x,
    y,
    alpha=0.5,
    s=30
)</code></pre>

<ul>
<li><code>x</code>, <code>y</code> — numeric coordinates.</li>
<li><code>alpha</code> — transparency; useful when points overlap.</li>
<li><code>s</code> — marker size.</li>
<li><code>c</code> — color or numeric values mapped to color.</li>
<li><code>marker</code> — marker style.</li>
</ul>

<p>Overplotting can hide density. Transparency, smaller markers, or jitter may help when many observations overlap.</p>
</details>

In [ ]:
plt.figure(figsize=(8, 4))

plt.scatter(
    orders["amount"],
    orders["satisfaction"],
    alpha=0.45
)

plt.xlabel("Order amount")
plt.ylabel("Satisfaction")
plt.title("Order Amount vs Satisfaction")
plt.yticks([1, 2, 3, 4, 5])
plt.tight_layout()
plt.show()

### Reading a scatter plot

Ask:

- Is there an upward pattern?
- Is there a downward pattern?
- Is there little visible pattern?
- Is the relationship approximately linear?
- Are there clusters?
- Are there unusual points?

Because satisfaction is recorded on a 1–5 scale, points appear in horizontal bands. That is expected.

## 14. Correlation

The Pearson correlation coefficient measures the direction and strength of a **linear** relationship.

It ranges from -1 to 1.

| Value | General interpretation |
|---:|---|
| close to 1 | strong positive linear relationship |
| close to -1 | strong negative linear relationship |
| close to 0 | weak linear relationship |

These descriptions are approximate. Context still matters.

<details>
<summary><strong>Syntax & parameters — corr()</strong></summary>

<pre><code class="language-python">df[["x", "y"]].corr()</code></pre>

<p><strong>Correlation methods</strong></p>
<pre><code class="language-python">df.corr(method="pearson")
df.corr(method="spearman")
df.corr(method="kendall")</code></pre>

<ul>
<li><code>pearson</code> — linear association between numeric variables.</li>
<li><code>spearman</code> — rank-based monotonic association.</li>
<li><code>kendall</code> — rank-based concordance measure.</li>
</ul>

<p>Correlation does not establish causation and can miss strong non-linear relationships.</p>
</details>

In [ ]:
correlation_matrix = orders[
    ["amount", "satisfaction"]
].corr()

correlation_matrix

In [ ]:
amount_satisfaction_corr = correlation_matrix.loc[
    "amount",
    "satisfaction"
]

print(
    "Correlation between amount and satisfaction:",
    round(amount_satisfaction_corr, 3)
)

### Important limitation

Satisfaction is an ordinal 1–5 score, so this correlation is best treated as a simple exploratory summary.

A correlation coefficient should always be interpreted together with the scatter plot and the meaning of the variables.

## 15. Correlation is not causation

A positive correlation between order amount and satisfaction does **not** prove that spending more causes satisfaction to increase.

Possible explanations include:

- another variable influences both;
- different customer groups behave differently;
- the sample may not represent the full population;
- the relationship may be partly coincidental.

A safer conclusion is:

> Order amount and satisfaction are positively associated in this dataset.

That describes the evidence without claiming causality.

## 16. Confidence interval: basic idea

Suppose these 400 orders are a sample from a larger population.

The sample mean is one estimate of the population mean.

A confidence interval provides a range of plausible values for that population mean.

For a reasonably large sample, a simple approximation is:

[
\text{mean} \pm 1.96 \times \text{standard error}
]

where

[
\text{standard error} = \frac{s}{\sqrt{n}}
]

This is an introductory approximation rather than a complete treatment of statistical inference.

<details>
<summary><strong>Syntax & concepts — standard error and confidence interval</strong></summary>

<pre><code class="language-python">mean = s.mean()
std = s.std(ddof=1)
n = s.count()

se = std / np.sqrt(n)
lower = mean - 1.96 * se
upper = mean + 1.96 * se</code></pre>

<p>The value <code>1.96</code> is the approximate 97.5th percentile of the standard normal distribution and is commonly used for a large-sample 95% interval.</p>

<p>For smaller samples or more formal inference, a t-based interval is often more appropriate.</p>

<p>Related SciPy tools include <code>scipy.stats.sem()</code> and <code>scipy.stats.t.interval()</code>.</p>
</details>

In [ ]:
import math

x = orders["amount"]

sample_mean = x.mean()
sample_std = x.std(ddof=1)
n = x.count()

standard_error = sample_std / math.sqrt(n)
margin = 1.96 * standard_error

lower = sample_mean - margin
upper = sample_mean + margin

print("Sample size:", n)
print("Sample mean:", round(sample_mean, 2))
print("Standard error:", round(standard_error, 2))
print(
    "Approximate 95% confidence interval:",
    round(lower, 2),
    "to",
    round(upper, 2)
)

### What the interval does not mean

The interval is about the **population mean**.

It does not mean that 95% of individual order amounts fall inside that interval.

That is a common misunderstanding.

## 17. Choose the right visualization

A chart should match the analytical question.

| Question | Useful visualization |
|---|---|
| What does one numeric distribution look like? | Histogram |
| What are the median and spread? | Boxplot |
| How do groups compare? | Grouped boxplot or bar chart |
| How are two numeric variables related? | Scatter plot |
| How does a measure change over time? | Line chart |

The best chart is the one that makes the question easiest to answer.

## 18. Good chart design

A useful chart should have:

- a clear title;
- meaningful axis labels;
- readable category labels;
- appropriate scales;
- enough space to read the chart;
- minimal unnecessary decoration.

A visualization should clarify the analysis, not decorate it.

## 19. Time-based analysis

When data contains dates, we often aggregate records by time period.

Instead of plotting every individual order, we can calculate daily revenue.

<details>
<summary><strong>Syntax & parameters — grouping and plotting time data</strong></summary>

<pre><code class="language-python">daily = (
    df.groupby("order_date", as_index=False)["amount"]
      .sum()
      .sort_values("order_date")
)</code></pre>

<p>Alternative with datetime resampling:</p>
<pre><code class="language-python">daily = (
    df.resample("D", on="order_date")["amount"]
      .sum()
      .reset_index()
)</code></pre>

<p>Always sort time values before drawing a line chart. Otherwise the line may connect dates in the wrong order.</p>
</details>

In [ ]:
daily_revenue = (
    orders
    .groupby("order_date", as_index=False)["amount"]
    .sum()
    .sort_values("order_date")
)

daily_revenue.head()

In [ ]:
plt.figure(figsize=(10, 4))

plt.plot(
    daily_revenue["order_date"],
    daily_revenue["amount"]
)

plt.xlabel("Date")
plt.ylabel("Daily revenue")
plt.title("Daily Revenue Trend")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Interpreting a line chart

Look for:

- upward or downward trends;
- temporary peaks or dips;
- repeated patterns;
- sudden changes.

A line chart is especially useful when the x-axis has a meaningful order, such as time.

## 20. Smooth a noisy time series

Daily data can be noisy.

A rolling average smooths short-term fluctuations and can make the broader pattern easier to see.

<details>
<summary><strong>Syntax & parameters — rolling averages</strong></summary>

<pre><code class="language-python">s.rolling(
    window=7,
    min_periods=None,
    center=False
).mean()</code></pre>

<ul>
<li><code>window</code> — number of observations in the moving window.</li>
<li><code>min_periods</code> — minimum observations needed to calculate a value.</li>
<li><code>center=True</code> — center the result on the window instead of using the right edge.</li>
</ul>

<p>Other rolling summaries include <code>sum()</code>, <code>median()</code>, <code>std()</code>, <code>min()</code>, and <code>max()</code>.</p>
</details>

In [ ]:
daily_revenue["rolling_7_day"] = (
    daily_revenue["amount"]
    .rolling(window=7)
    .mean()
)

daily_revenue.head(10)

In [ ]:
plt.figure(figsize=(10, 4))

plt.plot(
    daily_revenue["order_date"],
    daily_revenue["amount"],
    alpha=0.45,
    label="Daily revenue"
)

plt.plot(
    daily_revenue["order_date"],
    daily_revenue["rolling_7_day"],
    linewidth=2,
    label="7-day rolling average"
)

plt.xlabel("Date")
plt.ylabel("Revenue")
plt.title("Daily Revenue with 7-Day Rolling Average")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

The rolling average does not replace the original data.

It is another view of the same time series.

## 21. Build a small statistics report

A useful analytical report usually combines:

1. a clear question;
2. numerical summaries;
3. a visualization;
4. a short interpretation.

Example question:

> How does order amount differ across regions?

<details>
<summary><strong>Syntax guide — combining statistics and visualization</strong></summary>

<p>A compact report often follows this pattern:</p>

<pre><code class="language-python">summary = (
    df.groupby("group")["value"]
      .agg(["count", "mean", "median", "std"])
)

# then create a plot using the same grouping
</code></pre>

<p>A strong report should make the numerical table and chart answer the same analytical question.</p>

<p>When possible, state the unit, sample size, central tendency, spread, and any important limitation.</p>
</details>

In [ ]:
region_report = (
    orders
    .groupby("region")["amount"]
    .agg(
        orders="count",
        mean="mean",
        median="median",
        std="std"
    )
    .round(2)
)

region_report

In [ ]:
orders.boxplot(
    column="amount",
    by="region",
    grid=False,
    figsize=(8, 4)
)

plt.suptitle("")
plt.title("Order Amount by Region")
plt.xlabel("Region")
plt.ylabel("Order amount")
plt.tight_layout()
plt.show()

## 22. Write an interpretation

A strong interpretation connects the numerical summary with the visualization.

Useful questions include:

- Which region has the highest mean order amount?
- Is its median also high?
- Which region has the greatest variability?
- Are sample sizes similar?
- Does the boxplot support the same story as the table?

Use careful language such as:

- “higher on average,”
- “associated with,”
- “the data suggests,”
- “in this sample.”

Avoid causal language unless the study design actually supports a causal conclusion.

In [ ]:
highest_mean_region = region_report["mean"].idxmax()
highest_mean_value = region_report.loc[
    highest_mean_region,
    "mean"
]

print(
    "Highest mean order amount:",
    highest_mean_region,
    "->",
    highest_mean_value
)

print(
    "Amount/satisfaction correlation:",
    round(amount_satisfaction_corr, 3)
)

## Practice challenge

Use the `orders` DataFrame.

1. Display descriptive statistics for `amount`.
2. Calculate mean, median, standard deviation, and IQR.
3. Create a histogram of `amount`.
4. Create a boxplot of `amount`.
5. Summarize `amount` by region.
6. Create a grouped boxplot by region.
7. Create a bar chart of average amount by channel.
8. Plot amount versus satisfaction.
9. Calculate their correlation.
10. Create a line chart of daily total revenue.
11. Add a 7-day rolling average.
12. Write two or three sentences interpreting one result without claiming causation.

<details>
<summary><strong>Hints & syntax — practice challenge</strong></summary>

<p>Useful methods and functions:</p>
<pre><code class="language-python">describe()
mean()
median()
std()
quantile()
plt.hist()
plt.boxplot()
groupby().agg()
value_counts()
plt.scatter()
corr()
rolling().mean()</code></pre>

<p>For the written interpretation, describe what the evidence shows without turning association into causation.</p>
</details>

In [ ]:
# 1. Descriptive statistics


# 2. Mean, median, standard deviation, and IQR


# 3. Histogram


# 4. Boxplot


# 5. Summary by region


# 6. Grouped boxplot


# 7. Bar chart by channel


# 8. Scatter plot


# 9. Correlation


# 10. Daily revenue line chart


# 11. Rolling average


# 12. Interpretation as comments


## Summary and self-check

You should now be able to use statistics and visualizations together rather than treating them as separate tasks.

Make sure you can explain:

- why mean and median can differ;
- what standard deviation and IQR measure;
- what information a histogram provides;
- what a boxplot summarizes;
- when a grouped boxplot is useful;
- when to use a bar chart instead of a histogram;
- what a scatter plot can reveal;
- what correlation measures;
- why correlation does not establish causation;
- what a confidence interval is estimating;
- why time data should usually be ordered before plotting;
- what a rolling average does.

The main analytical habit is simple: **calculate, visualize, and interpret together.**